 # Blotter Check & Integrity Verification Notebook (v12)



 Validates calculated blotter fields against underlying OHLCV and feature data,

 and performs step-by-step verification of agent observations against PyTorch model scalers.

 Supports multi-chunk walk-forward models, canonical provenance slices, and dynamic metadata lineage.

In [ ]:
import json
import math
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from core.accounting import MTMPortfolioEngine
from core.logic import SelectionLogic
from core.paths import LOCAL_DATA_DIR, OUTPUT_DIR
from core.quant import QuantUtils
from core.settings import CacheConfig, TradingConfig, extract_run_hyperparameters
from data_pipeline.loader import load_processed_data

# Ensure project root is on sys.path dynamically
try:
    notebook_dir = Path(__file__).resolve().parent
except NameError:
    notebook_dir = Path.cwd()

if str(notebook_dir) not in sys.path:
    sys.path.append(str(notebook_dir))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

config = TradingConfig()

# Standardized folder layout references
ANCHORS_DIR = OUTPUT_DIR / "canonical_anchors"
CHECKPOINT_DIR = OUTPUT_DIR / "model_checkpoints"
CANONICAL_SLICE_PATH = ANCHORS_DIR / "canonical_scoring_slice.parquet"

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades



In [ ]:
# -----------------------------------------------------------------------------
# USER INPUT CONFIGURATION
# -----------------------------------------------------------------------------
# 1. Target Blotter: Path, filename string, or None (defaults to latest constituent seed blotter)
USER_SELECTED_FILE: str | int | Path | None = None

# 2. Audit Target Date: Set to canonical anchor date "2026-07-09" for Tier 1 bit-level audit,
#    or None to evaluate the terminal row (sample_idx = -1).
AUDIT_TARGET_DATE: str | None = None

# 1. Load Market Data Pipelines & Reward Matrix
data = load_processed_data()
df_ohlcv = data.df_ohlcv
macro_df = data.macro_df
features_df = data.features_df

# Pivot Close prices and compute 1-Day forward simple returns reward matrix
df_close = df_ohlcv["Adj Close"].unstack(level=0)
if config.handle_zeros_as_nan:
    df_close = df_close.replace(0.0, np.nan).ffill()
else:
    df_close = df_close.ffill(limit=config.max_data_gap_ffill)

reward_matrix = QuantUtils.build_forward_return_matrix(df_close, horizon=1)

# 2. Discover Blotter Parquet Artifacts in canonical_anchors and output
candidate_search_dirs = [ANCHORS_DIR, OUTPUT_DIR]
discovered_blotters: list[Path] = []

for s_dir in candidate_search_dirs:
    if s_dir.exists():
        found = [
            p
            for p in s_dir.glob("blotter_*.parquet")
            if "canonical_scoring_slice" not in p.name
        ]
        discovered_blotters.extend(found)

# Deduplicate preserving latest modification timestamp order
seen_paths = set()
unique_blotters: list[Path] = []
for p in sorted(discovered_blotters, key=lambda x: x.stat().st_mtime, reverse=True):
    if p.resolve() not in seen_paths:
        seen_paths.add(p.resolve())
        unique_blotters.append(p)

if not unique_blotters:
    raise FileNotFoundError(
        f"❌ No blotter parquet files found in {[str(d) for d in candidate_search_dirs if d.exists()]}"
    )

print("📁 Discovered Blotter Catalog:")
for idx, b_path in enumerate(unique_blotters):
    size_kb = b_path.stat().st_size / 1024
    print(f"   [{idx:2d}] {b_path.parent.name}/{b_path.name:<45} ({size_kb:6.1f} KB)")

# 3. Resolve Target Blotter File
selected_blotter_path: Path | None = None

if isinstance(USER_SELECTED_FILE, int):
    if 0 <= USER_SELECTED_FILE < len(unique_blotters):
        selected_blotter_path = unique_blotters[USER_SELECTED_FILE]
    else:
        raise IndexError(
            f"❌ Index {USER_SELECTED_FILE} out of bounds for blotter catalog (0..{len(unique_blotters)-1})"
        )
elif USER_SELECTED_FILE:
    candidate_path = Path(USER_SELECTED_FILE)
    if candidate_path.is_file():
        selected_blotter_path = candidate_path
    elif (ANCHORS_DIR / USER_SELECTED_FILE).is_file():
        selected_blotter_path = ANCHORS_DIR / USER_SELECTED_FILE
    elif (OUTPUT_DIR / USER_SELECTED_FILE).is_file():
        selected_blotter_path = OUTPUT_DIR / USER_SELECTED_FILE
    else:
        matches = [p for p in unique_blotters if USER_SELECTED_FILE in p.name]
        if not matches:
            raise FileNotFoundError(
                f"❌ User-specified blotter '{USER_SELECTED_FILE}' not found in canonical_anchors or output."
            )
        selected_blotter_path = matches[0]
else:
    # Default behavior: Prioritize constituent seed blotters (which have full 31-dim execution telemetry)
    seed_blotters = [p for p in unique_blotters if re.search(r"_s\d+_", p.name)]
    selected_blotter_path = seed_blotters[0] if seed_blotters else unique_blotters[0]

📁 Discovered Blotter Catalog:
   [ 0] canonical_anchors/blotter_continuous_ex_post_blend_gen25.parquet ( 776.8 KB)
   [ 1] output/blotter_continuous_ex_post_blend_gen25.parquet ( 776.8 KB)
   [ 2] output/blotter_continuous_s777_gen25.parquet         ( 773.1 KB)
   [ 3] output/blotter_continuous_s101_gen25.parquet         ( 773.5 KB)
   [ 4] output/blotter_continuous_s42_gen25.parquet          ( 775.0 KB)
   [ 5] canonical_anchors/blotter_continuous_ex_post_blend_gen24.parquet ( 773.9 KB)
   [ 6] output/blotter_continuous_ex_post_blend_gen24.parquet ( 773.9 KB)
   [ 7] output/blotter_continuous_s777_gen24.parquet         ( 773.4 KB)
   [ 8] output/blotter_continuous_s101_gen24.parquet         ( 774.5 KB)
   [ 9] output/blotter_continuous_s42_gen24.parquet          ( 771.9 KB)
   [10] output/blotter_continuous_ex_post_blend_gen23.parquet (  76.0 KB)
   [11] output/blotter_continuous_ex_post_blend_gen22.parquet (  75.4 KB)
   [12] canonical_anchors/blotter_continuous_ex_post_blend_gen23.p

In [ ]:
print(
    f"\n🎯 Selected Blotter Artifact: {selected_blotter_path.parent.name}/{selected_blotter_path.name}"
)
blotter_df = pd.read_parquet(selected_blotter_path)
run_stem = selected_blotter_path.stem

# Standardize date indexing column
if "date" not in blotter_df.columns and "decision_date" in blotter_df.columns:
    blotter_df["date"] = pd.to_datetime(blotter_df["decision_date"])
elif "date" in blotter_df.columns:
    blotter_df["date"] = pd.to_datetime(blotter_df["date"])

# 4. Resolve Generational Metadata & Run Hyperparameters
metadata: dict = {}
gen_match = re.search(r"gen(\d+)", selected_blotter_path.name)
gen_id = gen_match.group(1) if gen_match else None

candidate_meta_paths: list[Path] = []
if gen_id:
    candidate_meta_paths.extend(
        [
            CHECKPOINT_DIR / f"walk_forward_gen{gen_id}" / "run_metadata.json",
            CHECKPOINT_DIR
            / f"walk_forward_gen{gen_id}"
            / f"run_metadata_gen{gen_id}.json",
            ANCHORS_DIR / f"run_metadata_gen{gen_id}.json",
            OUTPUT_DIR / f"run_metadata_gen{gen_id}.json",
            OUTPUT_DIR / f"walk_forward_gen{gen_id}" / "run_metadata.json",
            selected_blotter_path.parent / f"run_metadata_gen{gen_id}.json",
        ]
    )

candidate_meta_paths.extend(
    [
        selected_blotter_path.parent / "run_metadata.json",
        ANCHORS_DIR / "run_metadata.json",
        OUTPUT_DIR / "run_metadata.json",
    ]
)

for m_cand in candidate_meta_paths:
    if m_cand.is_file():
        try:
            with open(m_cand, "r", encoding="utf-8") as f:
                metadata = json.load(f)
            print(f"📋 Loaded Run Metadata: {m_cand}")
            break
        except Exception as e:
            print(f"⚠️ Error reading {m_cand}: {e}")

run_params = extract_run_hyperparameters(
    selected_blotter_path.name, metadata, config, blotter_df=blotter_df
)

# Enforce dynamic configuration fallbacks (no hardcoded SPY or holding periods)
run_params.setdefault("holding_period", config.holding_period)
run_params.setdefault("min_basket_width", getattr(config, "min_basket_width", 5))
run_params.setdefault("max_cash_pct", getattr(config, "max_cash_pct", 0.0))
run_params.setdefault("min_active_tilt", getattr(config, "min_active_tilt", 0.20))
run_params.setdefault("upside_alpha_mult", getattr(config, "upside_alpha_mult", 1.0))
run_params.setdefault(
    "loss_aversion_penalty", getattr(config, "loss_aversion_penalty", 0.0)
)
run_params.setdefault("slippage_rate", getattr(config, "slippage_rate", 0.0010))
run_params.setdefault("benchmark", config.benchmark)


def extract_cache_filename(meta: dict) -> str | None:
    """Dynamically extracts the authoritative feature cache filename from metadata."""
    if not isinstance(meta, dict):
        return None
    # 1. Search nested data_lineage mapping
    data_lineage = meta.get("data_lineage", {})
    if isinstance(data_lineage, dict):
        for key in ("cache_file", "cache_filename", "feature_cache"):
            val = data_lineage.get(key)
            if val:
                return str(val)
    # 2. Search root-level metadata keys
    for key in ("cache_file", "cache_filename", "feature_cache"):
        val = meta.get(key)
        if val:
            return str(val)
    return None


# 5. Dynamically Resolve Feature Cache Lineage Matching Selected Generation
def resolve_cache_for_blotter(
    blotter_name: str, meta: dict
) -> tuple[pd.DataFrame, Path]:
    """Dynamically resolves the exact feature cache lineage matching the generation."""
    cache_name = extract_cache_filename(meta)
    if not cache_name:
        cache_name = CacheConfig.get_filename()

    cache_path = LOCAL_DATA_DIR / cache_name
    if not cache_path.exists():
        fallback_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
        if fallback_path.exists():
            print(
                f"⚠️ Metadata cache '{cache_name}' not found at {cache_path}. Falling back to default: {fallback_path.name}"
            )
            cache_path = fallback_path
        else:
            raise FileNotFoundError(
                f"❌ Cache '{cache_path}' specified in metadata and default cache '{fallback_path}' not found in {LOCAL_DATA_DIR}"
            )

    print(f"📦 Lineage Binding: Loading {cache_path.name} for {blotter_name}...")
    df = pd.read_parquet(cache_path)
    print(f"✅ Cache Loaded: {cache_path.name} ({len(df):,} rows)")
    return df, cache_path


alpha_df, alpha_cache_path = resolve_cache_for_blotter(
    selected_blotter_path.name, metadata
)

print(
    f"Total Tickers in df_ohlcv: {len(df_ohlcv.index.get_level_values('Ticker').unique())}"
)
print(f"Reward Matrix Shape:       {reward_matrix.shape}")
print(f"Resolved Hyperparameters: {run_params}")


🎯 Selected Blotter Artifact: output/blotter_continuous_s777_gen25.parquet
📋 Loaded Run Metadata: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output\model_checkpoints\walk_forward_gen25\run_metadata.json
📦 Lineage Binding: Loading alpha_cache_41d_2015.parquet for blotter_continuous_s777_gen25.parquet...
✅ Cache Loaded: alpha_cache_41d_2015.parquet (2,682,336 rows)
Total Tickers in df_ohlcv: 1710
Reward Matrix Shape:       (7480, 1710)
Resolved Hyperparameters: {'min_basket_width': 5, 'min_active_tilt': 0.2, 'upside_alpha_mult': 1.0, 'loss_aversion_penalty': 0.0, 'max_cash_pct': 0.0, 'slippage_rate': 0.001, 'holding_period': 5, 'gamma': 0.9, 'benchmark': 'SPY'}


In [ ]:
blotter_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1120 entries, 0 to 1119
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   date                              1120 non-null   datetime64[ns]
 1   decision_date                     1120 non-null   object        
 2   buy_date                          1120 non-null   object        
 3   sell_date                         1120 non-null   object        
 4   universe_size                     1120 non-null   int64         
 5   max_score                         1120 non-null   float64       
 6   min_score                         1120 non-null   float64       
 7   observation                       1120 non-null   object        
 8   raw_actions                       1120 non-null   object        
 9   offset                            1120 non-null   int64         
 10  width                             1120 non-null 

In [ ]:
# Target Date or Normalized Positional Resolution
if AUDIT_TARGET_DATE is not None:
    target_dt_ts = pd.Timestamp(AUDIT_TARGET_DATE)
    date_matches = blotter_df.index[blotter_df["date"] == target_dt_ts].tolist()
    if not date_matches:
        print(
            f"⚠️ Specified AUDIT_TARGET_DATE '{AUDIT_TARGET_DATE}' not found in blotter. Falling back to terminal row."
        )
        pos_idx = len(blotter_df) - 1
        sample_idx = -1
    else:
        pos_idx = date_matches[0]
        sample_idx = pos_idx
else:
    sample_idx = -1
    pos_idx = sample_idx % len(blotter_df)

blotter = blotter_df.iloc[pos_idx]


def get_field(row, *candidates, default="N/A"):
    for c in candidates:
        if c in row:
            val = row[c]
            if val is None:
                continue
            if isinstance(val, (list, tuple, np.ndarray)):
                if len(val) > 0:
                    return val
            elif pd.notna(val):
                return val
    return default


# Extract telemetry supporting both canonical 31-col and compact 7-col schemas
decision_date = get_field(blotter, "decision_date", "date")
buy_date = get_field(blotter, "buy_date")
sell_date = get_field(blotter, "sell_date")
universe_size = get_field(blotter, "universe_size")
max_score = get_field(blotter, "max_score")
min_score = get_field(blotter, "min_score")
observation = get_field(blotter, "observation", "obs", "state")
raw_actions = get_field(blotter, "raw_actions", "action", "actions")
decoded_offset = get_field(blotter, "offset", "decoded_offset")
decoded_width = get_field(blotter, "width", "decoded_width")
top_3_tickers = get_field(blotter, "top_3", "top_3_tickers")
chosen_tickers = get_field(
    blotter, "tickers", "chosen_tickers", "selected", "selected_tickers"
)
predicted_reward = get_field(blotter, "predicted_reward")
rl_reward_received = get_field(
    blotter, "rl_reward_received", "penalized_alpha_daily_simple_ret", "penalized_alpha"
)
actual_return = get_field(
    blotter, "net_daily_simple_ret", "actual_return", "portfolio_impact"
)
mkt_return = get_field(blotter, "bm_daily_simple_ret", "mkt_return")
cash_return = get_field(blotter, "cash_daily_simple_ret", "cash_return", default=0.0)
alpha = get_field(blotter, "alpha_daily_simple_ret", "alpha")
penalized_alpha = get_field(
    blotter, "penalized_alpha_daily_simple_ret", "penalized_alpha"
)
slippage_applied = get_field(blotter, "slippage_daily_simple_loss", "slippage_applied")
portfolio_impact = get_field(
    blotter, "net_daily_simple_ret", "portfolio_impact", "actual_return"
)
alpha_impact = get_field(
    blotter, "penalized_alpha_daily_simple_ret", "alpha_impact", "penalized_alpha"
)
agent_equity = get_field(blotter, "agent_equity", "equity", "portfolio_equity")
bm_equity = get_field(blotter, "bm_equity", "benchmark_equity")
alpha_equity = get_field(blotter, "alpha_equity")

print(
    f"\nSelected Blotter Row [sample_idx = {sample_idx} -> pos_idx = {pos_idx} / {len(blotter_df) - 1}]:"
)
print(blotter)


Selected Blotter Row [sample_idx = -1 -> pos_idx = 1119 / 1119]:
date                                                              2026-09-23 00:00:00
decision_date                                                              2026-09-23
buy_date                                                                   2026-09-24
sell_date                                                                  2026-09-25
universe_size                                                                    1024
max_score                                                                    5.347118
min_score                                                                   -3.131586
observation                         [0.26493385434150696, -0.6638742685317993, -0....
raw_actions                         [-0.3283427357673645, -0.0451325885951519, 0.0...
offset                                                                            332
width                                                                     

In [ ]:
def parse_tickers(val):
    """Safe parser for lists, ndarrays, and serialized strings."""
    if isinstance(val, (list, tuple, np.ndarray)):
        return [str(t).strip() for t in val if pd.notna(t) and str(t).strip()]
    elif isinstance(val, str):
        cleaned = (
            val.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        )
        return [t.strip() for t in cleaned.split(",") if t.strip()]
    return []


parsed_chosen = parse_tickers(chosen_tickers)
print("--- 1. Ticker Parsing Check ---")
print(f"Parsed chosen tickers: {parsed_chosen}")

print("\n--- 2. Date Alignment Check ---")
print(f"decision_date: {decision_date}")
print(f"buy_date:      {buy_date}")
print(f"sell_date:     {sell_date}")

--- 1. Ticker Parsing Check ---
Parsed chosen tickers: ['CDW', 'HBAN', 'AXP', 'CNM', 'TJX', 'AMD']

--- 2. Date Alignment Check ---
decision_date: 2026-09-23
buy_date:      2026-09-24
sell_date:     2026-09-25


In [ ]:
decision_date_ts = pd.Timestamp(decision_date)

# 1. Resolve Active Cross-Sectional Alpha Cube (Canonical Slice vs Live Cache)
active_feature_cube = alpha_df
is_canonical_slice_active = False

if CANONICAL_SLICE_PATH.is_file():
    try:
        slice_df = pd.read_parquet(CANONICAL_SLICE_PATH)
        slice_dates = slice_df.index.get_level_values("Date").unique()
        if decision_date_ts in slice_dates:
            active_feature_cube = slice_df
            is_canonical_slice_active = True
            print(
                f"🏛️ Bound to Canonical Provenance Slice for {decision_date_ts.strftime('%Y-%m-%d')} (Hermetic Bit-Level Anchor)"
            )
    except Exception as e:
        print(f"ℹ️ Canonical slice probe note: {e}")

if "Date" in active_feature_cube.index.names:
    dates_idx = active_feature_cube.index.get_level_values("Date")
    if decision_date_ts in dates_idx:
        alpha_result = active_feature_cube.xs(decision_date_ts, level="Date")
    elif decision_date_ts.strftime("%Y-%m-%d") in dates_idx:
        alpha_result = active_feature_cube.xs(
            decision_date_ts.strftime("%Y-%m-%d"), level="Date"
        )
    else:
        alpha_result = pd.DataFrame()
elif decision_date_ts in active_feature_cube.index:
    alpha_result = active_feature_cube.loc[decision_date_ts]
else:
    alpha_result = pd.DataFrame()

cal_universe_size = len(alpha_result) if not alpha_result.empty else universe_size

# 2. Decode Action via Stateless SelectionLogic Engine
cal_decoded_offset = decoded_offset
cal_decoded_width = decoded_width
cal_equity_exposure = float(get_field(blotter, "equity_exposure", default=1.0))
cal_active_tilt = float(get_field(blotter, "active_tilt", default=1.0))

has_raw_actions = (
    raw_actions is not None
    and isinstance(raw_actions, (list, tuple, np.ndarray))
    and len(raw_actions) > 0
)

if (
    has_raw_actions
    and isinstance(alpha_result, pd.DataFrame)
    and not alpha_result.empty
):
    try:
        (
            _,
            _,
            cal_decoded_offset,
            cal_decoded_width,
            cal_equity_exposure,
            cal_active_tilt,
            _,
            _,
        ) = SelectionLogic.apply_action(
            ensemble=alpha_result,
            action=np.asarray(raw_actions, dtype=np.float64),
            rank_max_offset_percentile=config.rank_max_offset_percentile,
            rank_max_width=config.rank_max_width,
            min_basket_width=run_params["min_basket_width"],
            max_cash_pct=run_params["max_cash_pct"],
            min_active_tilt=run_params["min_active_tilt"],
        )
    except Exception as e:
        print(f"ℹ️ Action decoding note: {e}")

# 3. Verify 1-Day Benchmark & Cash Returns
benchmark_ticker = run_params.get(
    "benchmark",
    getattr(config, "benchmark", config.benchmark),
)
if (
    decision_date_ts in reward_matrix.index
    and benchmark_ticker in reward_matrix.columns
):
    bm_val = reward_matrix.loc[decision_date_ts, benchmark_ticker]
    bm_raw = float(np.asarray(bm_val).ravel()[0])
    cal_mkt_return = float(bm_raw) if np.isfinite(bm_raw) else 0.0
else:
    cal_mkt_return = float(
        get_field(blotter, "bm_daily_simple_ret", "mkt_return", default=0.0)
    )

cal_cash_return = float(
    get_field(blotter, "cash_daily_simple_ret", "cash_return", default=0.0)
)

# 4. Reconstruct Rolling FIFO Active Sleeves with Strict T+1 Close Execution
holding_period = int(run_params["holding_period"])
tickers_col = next(
    (
        c
        for c in ["tickers", "chosen_tickers", "selected", "selected_tickers"]
        if c in blotter_df.columns
    ),
    None,
)

if tickers_col and pos_idx > 0:
    executing_sleeve = parse_tickers(blotter_df.iloc[pos_idx - 1][tickers_col])
    active_start_idx = max(0, pos_idx - holding_period)
    active_sleeves_history = blotter_df.iloc[active_start_idx:pos_idx][
        tickers_col
    ].tolist()
else:
    executing_sleeve = None
    active_sleeves_history = []

# 5. Verify 1-Day MTM Active Stock Return using forward reward_matrix
num_active = len(active_sleeves_history)
if num_active > 0:
    sleeve_daily_returns = []
    for sleeve_tickers in active_sleeves_history:
        chosen_list = parse_tickers(sleeve_tickers)
        if len(chosen_list) > 0 and decision_date_ts in reward_matrix.index:
            sleeve_ret = MTMPortfolioEngine.calculate_sleeve_simple_ret(
                chosen_list, reward_matrix.loc[decision_date_ts]
            )
            sleeve_daily_returns.append(sleeve_ret)
        else:
            sleeve_daily_returns.append(0.0)

    active_sum = float(np.sum(sleeve_daily_returns))
    unfilled_count = holding_period - num_active
    cal_raw_stock_return = (
        active_sum + (unfilled_count * cal_mkt_return)
    ) / holding_period
else:
    cal_raw_stock_return = cal_mkt_return

# 6. Verify Tri-Asset Allocation Weights
current_chosen = parse_tickers(chosen_tickers)
if num_active > 0 or len(current_chosen) > 0:
    cal_w_active = cal_equity_exposure * cal_active_tilt
    cal_w_benchmark = cal_equity_exposure * (1.0 - cal_active_tilt)
    cal_w_cash = max(0.0, 1.0 - (cal_w_active + cal_w_benchmark))
else:
    cal_w_active = 0.0
    cal_w_benchmark = cal_equity_exposure
    cal_w_cash = max(0.0, 1.0 - (cal_w_active + cal_w_benchmark))

# 7. Verify Gross Return, Amortized Slippage & Net Actual Return
if tickers_col:
    cal_gross_return = (
        cal_w_active * cal_raw_stock_return
        + cal_w_benchmark * cal_mkt_return
        + cal_w_cash * cal_cash_return
    )
    slippage_rate = float(
        getattr(config, "slippage_rate", run_params.get("slippage_rate", 0.0010))
    )
    cal_slippage_applied = (
        float((slippage_rate / holding_period) * cal_w_active)
        if executing_sleeve is not None
        and len(executing_sleeve) > 0
        and cal_w_active > 0.0
        else 0.0
    )
    cal_actual_return = cal_gross_return - cal_slippage_applied
else:
    cal_actual_return = float(actual_return)
    cal_slippage_applied = float(slippage_applied) if slippage_applied != "N/A" else 0.0

cal_alpha = cal_actual_return - cal_mkt_return

# 8. Verify Penalized Alpha / RL Reward
upside_mult = float(run_params["upside_alpha_mult"])
loss_aversion_penalty = float(run_params["loss_aversion_penalty"])

if cal_alpha > 0.0:
    cal_penalized_alpha = cal_alpha * upside_mult
elif cal_alpha < 0.0:
    loss_multiplier = 1.0 + max(0.0, loss_aversion_penalty)
    cal_penalized_alpha = cal_alpha * loss_multiplier
else:
    cal_penalized_alpha = 0.0

if not np.isfinite(cal_penalized_alpha):
    cal_penalized_alpha = 0.0

cal_rl_reward_received = float(cal_penalized_alpha)
cal_portfolio_impact = cal_actual_return
cal_alpha_impact = cal_penalized_alpha

# 9. Verify Compounded Equity Curves
agent_eq_col = next(
    (
        c
        for c in ["agent_equity", "equity", "portfolio_equity"]
        if c in blotter_df.columns
    ),
    "agent_equity",
)
bm_eq_col = next(
    (c for c in ["bm_equity", "benchmark_equity"] if c in blotter_df.columns),
    "bm_equity",
)

if pos_idx == 0:
    prev_agent_equity = 1.0
    prev_benchmark_equity = 1.0
else:
    prev_agent_equity = float(blotter_df.iloc[pos_idx - 1][agent_eq_col])
    prev_benchmark_equity = float(blotter_df.iloc[pos_idx - 1][bm_eq_col])

# Strict Geometric Wealth Ratio (Base 1.0: V_p(t) / V_bm(t))
cal_agent_equity, cal_bm_equity, cal_alpha_equity = QuantUtils.step_compounding_curves(
    prev_portfolio_equity=prev_agent_equity,
    prev_benchmark_equity=prev_benchmark_equity,
    net_daily_ret=cal_actual_return,
    bm_daily_ret=cal_mkt_return,
)

In [ ]:
verification_records = [
    ("universe_size", universe_size, cal_universe_size),
    ("mkt_return", mkt_return, cal_mkt_return),
    ("actual_return", actual_return, cal_actual_return),
    ("alpha", alpha, cal_alpha),
    ("penalized_alpha", penalized_alpha, cal_penalized_alpha),
    ("rl_reward_received", rl_reward_received, cal_rl_reward_received),
    ("slippage_applied", slippage_applied, cal_slippage_applied),
    ("portfolio_impact", portfolio_impact, cal_portfolio_impact),
    ("alpha_impact", alpha_impact, cal_alpha_impact),
    ("agent_equity", agent_equity, cal_agent_equity),
    ("alpha_equity", alpha_equity, cal_alpha_equity),
    ("decoded_offset", decoded_offset, cal_decoded_offset),
    ("decoded_width", decoded_width, cal_decoded_width),
]

summary_df = pd.DataFrame(
    verification_records,
    columns=["Metric", "Blotter Value", "Calculated Value"],
)


def safe_match(row, tol=1e-6):
    metric = row["Metric"]
    val1 = row["Blotter Value"]
    val2 = row["Calculated Value"]

    # Tier 2 Restatement Tolerances for Universe Size & Offset
    if not is_canonical_slice_active and metric in ("universe_size", "decoded_offset"):
        u_blotter = float(universe_size)
        u_live = float(cal_universe_size)
        if abs(u_blotter - u_live) / u_blotter <= 0.05:
            if metric == "universe_size":
                return True
            if metric == "decoded_offset":
                pct_b = float(decoded_offset) / u_blotter
                pct_c = float(cal_decoded_offset) / u_live
                max_rank_tol = 1.0 / min(u_blotter, u_live)
                return abs(pct_b - pct_c) <= max_rank_tol

    try:
        a1 = np.asarray(val1)
        a2 = np.asarray(val2)
        if a1.ndim > 0 or a2.ndim > 0:
            if a1.shape == a2.shape:
                return bool(
                    np.allclose(
                        a1.astype(float), a2.astype(float), atol=tol, equal_nan=True
                    )
                )
            return False
        f1, f2 = float(a1), float(a2)
        if np.isnan(f1) and np.isnan(f2):
            return True
        return math.isclose(f1, f2, abs_tol=tol)
    except (ValueError, TypeError):
        return str(val1).strip() == str(val2).strip()


summary_df["Match"] = summary_df.apply(safe_match, axis=1)


def format_status(row):
    if not row["Match"]:
        return "❌ FAIL"
    if not is_canonical_slice_active and row["Metric"] in (
        "universe_size",
        "decoded_offset",
    ):
        if row["Blotter Value"] != row["Calculated Value"]:
            return "✅ PASS (Tier 2)"
    return "✅ PASS"


summary_df["Status"] = summary_df.apply(format_status, axis=1)

pd.set_option("display.float_format", lambda x: f"{x:.8f}")

print("======================================================================")
print(
    f" DAILY MTM BLOTTER VERIFICATION SUMMARY (Date: {decision_date_ts.strftime('%Y-%m-%d')}, pos_idx={pos_idx}, H={holding_period})"
)
print("======================================================================")
print(summary_df[["Metric", "Blotter Value", "Calculated Value", "Status"]])
print("======================================================================")

if summary_df["Match"].all():
    print("🎉 ALL DAILY MTM METRICS VERIFIED SUCCESSFULLY!")
else:
    failed_metrics = summary_df[~summary_df["Match"]]["Metric"].tolist()
    print(f"⚠️ VERIFICATION FAILED FOR METRICS: {failed_metrics}")

 DAILY MTM BLOTTER VERIFICATION SUMMARY (Date: 2026-09-23, pos_idx=1119, H=5)
                Metric  Blotter Value  Calculated Value  Status
0        universe_size  1024.00000000     1024.00000000  ✅ PASS
1           mkt_return    -0.00082052       -0.00082052  ✅ PASS
2        actual_return    -0.00518882       -0.00518882  ✅ PASS
3                alpha    -0.00436831       -0.00436831  ✅ PASS
4      penalized_alpha    -0.00436831       -0.00436831  ✅ PASS
5   rl_reward_received    -0.00436831       -0.00436831  ✅ PASS
6     slippage_applied     0.00017473        0.00017473  ✅ PASS
7     portfolio_impact    -0.00518882       -0.00518882  ✅ PASS
8         alpha_impact    -0.00436831       -0.00436831  ✅ PASS
9         agent_equity     2.46982550        2.46982550  ✅ PASS
10        alpha_equity     1.40838739        1.40838739  ✅ PASS
11      decoded_offset   332.00000000      332.00000000  ✅ PASS
12       decoded_width     6.00000000        6.00000000  ✅ PASS
🎉 ALL DAILY MTM METRICS VE

In [ ]:
def verify_agent_observation(
    blotter_source: Path | pd.DataFrame,
    pt_model_path: Path,
    target_date: pd.Timestamp,
    verbose: bool = True,
) -> float:
    """Reconstructs agent observations manually from raw cache & scaler state and verifies against recorded blotter values."""
    date_str = target_date.strftime("%Y-%m-%d")

    # 1. Load Raw Pipeline
    raw_data = load_processed_data()
    raw_macro_df = raw_data.macro_df

    # 2. Select Feature Cube (Prioritize Canonical Slice for hermetic provenance)
    feature_cube = alpha_df
    using_slice = False
    if CANONICAL_SLICE_PATH.is_file():
        try:
            cand_slice = pd.read_parquet(CANONICAL_SLICE_PATH)
            if target_date in cand_slice.index.get_level_values("Date"):
                feature_cube = cand_slice
                using_slice = True
        except Exception:
            pass

    # 3. Load Blotter Observation Slice
    if isinstance(blotter_source, pd.DataFrame):
        df_target = blotter_source.copy()
    else:
        df_target = pd.read_parquet(blotter_source)

    date_col = next(
        (c for c in ["date", "decision_date"] if c in df_target.columns), None
    )
    if not date_col:
        raise KeyError("❌ No date/decision_date column found in blotter_df.")

    df_target[date_col] = pd.to_datetime(df_target[date_col])

    obs_col = next(
        (
            col
            for col in ["observation", "obs", "state", "raw_obs"]
            if col in df_target.columns
        ),
        None,
    )
    if not obs_col:
        raise KeyError("❌ Could not find an observation column in blotter_df.")

    blotter_row = df_target[df_target[date_col] == target_date]
    if blotter_row.empty:
        raise ValueError(f"❌ Date {date_str} not found in blotter_df!")
    blotter_row = blotter_row.iloc[0]
    blotter_obs = np.asarray(blotter_row[obs_col], dtype=np.float64)

    if blotter_obs is None or blotter_obs.size == 0:
        if verbose:
            print("ℹ️ Target slice observation is empty. Skipping scaler match.")
        return 0.0

    # 4. Load Scaler Weights from PyTorch Model
    checkpoint = torch.load(pt_model_path, map_location="cpu", weights_only=False)
    scaler_state = checkpoint.get("scaler_state")
    if not scaler_state:
        raise ValueError(f"❌ 'scaler_state' not found inside {pt_model_path.name}")

    # 5. Reconstruct Multi-Dim Observation Manually
    expected_strats = len(feature_cube.columns)

    # Step A: Cross-Sectional Universe Mean & Std
    if "Date" in feature_cube.index.names:
        dates_cube = feature_cube.index.get_level_values("Date")
        if target_date in dates_cube:
            ensemble = feature_cube.xs(target_date, level="Date")
        elif date_str in dates_cube:
            ensemble = feature_cube.xs(date_str, level="Date")
        else:
            ensemble = pd.DataFrame()
    else:
        ensemble = (
            feature_cube.loc[target_date]
            if target_date in feature_cube.index
            else pd.DataFrame()
        )

    if not ensemble.empty and ensemble.shape[1] == expected_strats:
        strat_mean = np.nan_to_num(
            np.asarray(ensemble.mean(axis=0), dtype=float), nan=0.0
        )
        strat_std = np.nan_to_num(
            np.asarray(ensemble.std(axis=0, ddof=0), dtype=float), nan=0.0
        )
    else:
        strat_mean = np.zeros(expected_strats)
        strat_std = np.zeros(expected_strats)

    # Step B: Benchmark Feature Vector Slice
    bm_ticker = run_params.get(
        "benchmark",
        getattr(config, "benchmark", config.benchmark),
    )
    try:
        bm_cube = feature_cube.xs(bm_ticker, level="Ticker")
        if target_date in bm_cube.index:
            bm_vals = bm_cube.loc[target_date].fillna(0.0).values
        elif date_str in bm_cube.index:
            bm_vals = bm_cube.loc[date_str].fillna(0.0).values
        else:
            bm_vals = np.zeros(expected_strats)
    except KeyError:
        # Fallback to live cache if benchmark is pruned from slice
        try:
            bm_live = alpha_df.xs(bm_ticker, level="Ticker")
            bm_vals = (
                bm_live.loc[target_date].fillna(0.0).values
                if target_date in bm_live.index
                else np.zeros(expected_strats)
            )
        except Exception:
            bm_vals = np.zeros(expected_strats)

    # Step C: Macro Context Slice
    if target_date in raw_macro_df.index:
        macro_row = raw_macro_df.loc[target_date]
    elif date_str in raw_macro_df.index:
        macro_row = raw_macro_df.loc[date_str]
    else:
        macro_row = pd.Series(0.0, index=raw_macro_df.columns)
    macro_vals = macro_row.fillna(0.0).values

    # Step D: Assemble and Scale (Single-Precision Float32 Arithmetic)
    raw_obs = np.concatenate([strat_mean, strat_std, bm_vals, macro_vals]).astype(
        np.float32
    )
    raw_obs = np.nan_to_num(raw_obs, nan=0.0, posinf=0.0, neginf=0.0)

    mean = np.asarray(scaler_state["mean"], dtype=np.float32)
    var = np.asarray(scaler_state["var"], dtype=np.float32)
    count = float(scaler_state["count"])
    std = np.sqrt(var / count) + 1e-8
    manual_final_obs = np.arcsinh((raw_obs - mean) / std)

    # 6. Audit & Comparison (Float32 numerical precision threshold: 2e-5)
    diff = np.abs(manual_final_obs - blotter_obs)
    max_diff = float(np.max(diff))

    if verbose:
        print("\n" + "=" * 50)
        print(
            f"📊 OBSERVATION SCALER VERIFICATION ({pt_model_path.parent.name}/{pt_model_path.name})"
        )
        print("=" * 50)
        print(f"Target Session Date:   {date_str}")
        print(
            f"Provenance Source:     {'Canonical Slice (u=1024)' if using_slice else f'Live Cache (u={len(ensemble)})'}"
        )
        print(
            f"Observation Dimension: {len(manual_final_obs)} (Blotter: {len(blotter_obs)})"
        )
        print(f"Max Absolute Error:    {max_diff:.8e}")

        if max_diff <= 2e-5:
            print(
                "✅ SUCCESS: Observation verified down to bit-level float32 identity (atol <= 2e-5)!"
            )
        elif not using_slice:
            print(
                "ℹ️ NOTICE: Expected Tier 2 variance due to live delisting cache restatement."
            )
        else:
            print("❌ FAILED: Unexpected drift detected on canonical slice.")

        print("\n--- First 5 Dims Comparison ---")
        print(f"{'Dim':<5} | {'Manual Calculation':<20} | {'Blotter Recording'}")
        print("-" * 50)
        for i in range(min(5, len(manual_final_obs))):
            print(f"{i:<5} | {manual_final_obs[i]:<20.6f} | {blotter_obs[i]:.6f}")

    return max_diff

In [ ]:
is_blend_blotter = (
    "blend" in selected_blotter_path.name or "committee" in selected_blotter_path.name
)

if is_blend_blotter:
    print(
        f"\nℹ️ Notice: Selected artifact '{selected_blotter_path.name}' is an ex-post capital blend."
        f"\n   Observation scaler verification is designed for single-agent constituent seed runs."
    )
else:
    # 1. Extract Seed and Generation Tokens from Selected Blotter
    seed_match = re.search(r"s(\d+)", selected_blotter_path.name)
    gen_match = re.search(r"gen(\d+)", selected_blotter_path.name)

    seed_token = f"s{seed_match.group(1)}" if seed_match else ""
    gen_folder = f"walk_forward_gen{gen_match.group(1)}" if gen_match else ""

    candidate_checkpoints: list[Path] = []
    if CHECKPOINT_DIR.exists():
        if gen_folder and (CHECKPOINT_DIR / gen_folder).exists():
            candidate_checkpoints = sorted(
                (CHECKPOINT_DIR / gen_folder).glob(f"*{seed_token}*.pt")
            )
        else:
            candidate_checkpoints = sorted(
                CHECKPOINT_DIR.glob(f"*{gen_folder}*/*{seed_token}*.pt")
            )

    if not candidate_checkpoints:
        print(
            f"⚠️ No checkpoints found matching generation '{gen_folder}' and seed '{seed_token}'"
        )
    else:
        target_date = decision_date_ts
        num_chunks = len(candidate_checkpoints)

        # 2. Chronological Walk-Forward Fold Mapping
        chronological_chunk_idx = min(
            num_chunks - 1, int((pos_idx / max(1, len(blotter_df))) * num_chunks)
        )
        chronological_ckpt = candidate_checkpoints[chronological_chunk_idx]

        print(
            f"\n🧠 Walk-Forward Model Topology for {seed_token} in {gen_folder} ({num_chunks} chunks discovered):"
        )
        for idx, cp in enumerate(candidate_checkpoints):
            active_marker = (
                " 👈 (Active Chronological Fold)"
                if idx == chronological_chunk_idx
                else ""
            )
            print(f"   [{idx}] {cp.parent.name}/{cp.name}{active_marker}")

        # 3. Check for Bit-Level Match (< 2e-5) Across Candidates
        best_ckpt: Path | None = None
        min_error = float("inf")

        for cp in candidate_checkpoints:
            try:
                err = verify_agent_observation(
                    blotter_source=blotter_df,
                    pt_model_path=cp,
                    target_date=target_date,
                    verbose=False,
                )
                if err < min_error:
                    min_error = err

                if err <= 2e-5:
                    best_ckpt = cp
                    print(
                        f"\n🎯 Locked onto active walk-forward chunk (Bit-Level Identity): {cp.parent.name}/{cp.name}"
                    )
                    verify_agent_observation(
                        blotter_source=blotter_df,
                        pt_model_path=cp,
                        target_date=target_date,
                        verbose=True,
                    )
                    break
            except Exception:
                continue

        # 4. Fallback for Restated Cache: Use the true Chronological Chunk
        if not best_ckpt:
            print(
                f"\nℹ️ Live Restated Cache Active: Evaluating true chronological fold model '{chronological_ckpt.name}'..."
            )
            verify_agent_observation(
                blotter_source=blotter_df,
                pt_model_path=chronological_ckpt,
                target_date=target_date,
                verbose=True,
            )


🧠 Walk-Forward Model Topology for s777 in walk_forward_gen25 (5 chunks discovered):
   [0] walk_forward_gen25/model_chunk0_s777_champion.pt
   [1] walk_forward_gen25/model_chunk1_s777_champion.pt
   [2] walk_forward_gen25/model_chunk2_s777_champion.pt
   [3] walk_forward_gen25/model_chunk3_s777_champion.pt
   [4] walk_forward_gen25/model_chunk4_s777_champion.pt 👈 (Active Chronological Fold)

🎯 Locked onto active walk-forward chunk (Bit-Level Identity): walk_forward_gen25/model_chunk4_s777_champion.pt

📊 OBSERVATION SCALER VERIFICATION (walk_forward_gen25/model_chunk4_s777_champion.pt)
Target Session Date:   2026-09-23
Provenance Source:     Live Cache (u=1024)
Observation Dimension: 46 (Blotter: 46)
Max Absolute Error:    1.19209290e-07
✅ SUCCESS: Observation verified down to bit-level float32 identity (atol <= 2e-5)!

--- First 5 Dims Comparison ---
Dim   | Manual Calculation   | Blotter Recording
--------------------------------------------------
0     | 0.264934             | 0.264